# Impacto da renda média sobre a mortalidade no Brasil

## Objetivo

Investigar a associação entre renda média estadual e taxas de mortalidade
no Brasil utilizando dados públicos do Sistema de Informações sobre
Mortalidade (SIM), PNAD Contínua e estimativas populacionais do IBGE.

Hipótese:

Estados com menor renda média apresentam maiores taxas de mortalidade.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr

import statsmodels.api as sm
import statsmodels.formula.api as smf

#### Databases

In [2]:
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/'

sim = pd.read_parquet(f"{PATH}sim/standardization/sim_2018_2022.parquet")

popul = pd.read_parquet(f"{PATH}ibge_populacao/ibge_populacao2.parquet")

pnad = pd.read_parquet(f"{PATH}ibge_pnad_continua/ibge_pnad_continua.parquet")

ibge = pd.read_parquet(f"{PATH}ibge_codigos/02_standardization/dtb_br_2024_municipios.parquet")

### Construction of the analytical base

- Contagem de óbitos

In [3]:
sim = sim[["cod_mun_res", "ano"]]
#ibge = ibge[["cod_mun_comp", "cod_uf"]]
sim["cod_mun_res"] = pd.to_numeric(sim["cod_mun_res"], errors="coerce").astype("Int64")
sim["cod_uf"] = sim["cod_mun_res"] // 10000

In [8]:
obitos_uf = sim.groupby(["cod_uf", "ano"]).size().reset_index(name="obitos")

- População por UF

In [10]:
pop_uf = popul.merge(ibge[["cod_mun_comp", "cod_uf"]], left_on="cod_municipio", right_on="cod_mun_comp",how="left")

In [11]:
pop_cod_uf = (pop_uf.groupby(["cod_uf_x", "ano"])["populacao"].sum().reset_index())

In [21]:
pop_cod_uf = pop_cod_uf.rename(columns={"cod_uf_x" : "cod_uf"})

- Unindo com renda

In [30]:
obitos_uf["ano"] = obitos_uf["ano"].astype(str).str.extract(r"(\d{4})")[0]

In [32]:
obitos_uf["ano"] = obitos_uf["ano"].astype("Int64")
pop_cod_uf["ano"] = pop_cod_uf["ano"].astype("Int64")
pnad["ano"] = pnad["ano"].astype("Int64")

In [33]:
df = (obitos_uf.merge(pop_cod_uf, on=["cod_uf", "ano"]).merge(pnad, on=["cod_uf", "ano"]))

### Write

In [36]:
df_final.coalesce(1) \
  .write \
  .mode("overwrite") \
  .parquet("...")